# Libraries

In [20]:
import httpx
import time
import json
import pandas as pd
import numpy as np
from returns.result import safe
from nltk import word_tokenize
from tqdm import tqdm
from collections import defaultdict

In [2]:
BASE_URL = "http://0.0.0.0:8000/predict"
MIN_TOKENS, MAX_TOKENS = 16, 300
BATCH_SIZE = 64

# Utilities

In [3]:
@safe(exceptions=(httpx.TransportError,httpx.HTTPStatusError))
def make_request(
    url : str,
    documents : list[str],
    min_threshold : float = 0.3,
    k : int = 10,
    max_tries : int = 3
) -> httpx.Response:
    
    params = {
        'threshold': min_threshold,
        'k': k
    }

    documents = [
        {
            "description" : doc
        }
        for doc in documents
    ]

    body = {
        "documents" : documents
    }

    response = None

    for attempt in range(max_tries):

        try:

            response = httpx.post(url=url,params=params,json=body)

            if response.status_code in [
                httpx.codes.TOO_MANY_REQUESTS,
                httpx.codes.INTERNAL_SERVER_ERROR,
                httpx.codes.BAD_GATEWAY,
                httpx.codes.SERVICE_UNAVAILABLE,
                httpx.codes.GATEWAY_TIMEOUT
            ]:
                time.sleep(2 ** (attempt * 0.8))   
            else:
                break
            
        except httpx.TransportError as e:

            if attempt == (max_tries - 1):
                raise e
            
            time.sleep(2 ** (attempt * 0.8)) 

    return response.raise_for_status()

- Test request

In [4]:
result = make_request(
    url=BASE_URL,
    documents=["Ontologies have been known for their powerful semantic representation of knowledge. However, ontologies cannot automatically evolve to reflect updates that occur in respective domains. To address this limitation, researchers have called for automatic ontology generation from unstructured text corpus. Unfortunately, systems that aim to generate ontologies from unstructured text corpus are domain-specific and require manual intervention. In addition, they suffer from uncertainty in creating concept linkages and difficulty in finding axioms for the same concept. Knowledge Graphs (KGs) has emerged as a powerful model for the dynamic representation of knowledge. However, KGs have many quality limitations and need extensive refinement. This research aims to develop a novel domain-independent automatic ontology generation framework that converts unstructured text corpus into domain consistent ontological form. The framework generates KGs from unstructured text corpus as well as refine and correct them to be consistent with domain ontologies. The power of the proposed automatically generated ontology is that it integrates the dynamic features of KGs and the quality features of ontologies"],
    min_threshold=0.0,
    k=10
)

In [5]:
result.unwrap().json()

[[{'name': 'info.info-ai', 'rank': 0},
  {'name': 'info.info-wb', 'rank': 1},
  {'name': 'info.info-ir', 'rank': 2},
  {'name': 'info.info-cl', 'rank': 3},
  {'name': 'info.info-db', 'rank': 4},
  {'name': 'info.info-tt', 'rank': 5},
  {'name': 'info.info-si', 'rank': 6},
  {'name': 'info.info-mm', 'rank': 7},
  {'name': 'info.info-oh', 'rank': 8},
  {'name': 'info.info-lg', 'rank': 9}]]

# Load data

In [6]:
data = pd.read_csv('subset.csv')
data.head()

,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed,year
0,2401.03814,Raul Infante-Sainz,"Ra\'ul Infante-Sainz, Mohammad Akhlaghi",Gnuastro: visualizing the full dynamic range i...,Accepted RNAAS. Supplementary data on Zenodo\n...,NaN,NaN,NaN,astro-ph.IM astro-ph.GA cs.CV,http://creativecommons.org/licenses/by-sa/4.0/,Color plays a crucial role in the visualizat...,"[{'version': 'v1', 'created': 'Mon, 8 Jan 2024...",2024-01-09,"[['Infante-Sainz', 'Raúl', ''], ['Akhlaghi', '...",2024
1,2309.02388,Zhibao Zheng,"Zhibao Zheng, David N\'eron, Udo Nackenhorst",A stochastic LATIN method for stochastic and p...,NaN,NaN,NaN,NaN,math.NA cs.NA,http://creativecommons.org/licenses/by-nc-nd/4.0/,The LATIN method has been developed and succ...,"[{'version': 'v1', 'created': 'Tue, 5 Sep 2023...",2023-09-06,"[['Zheng', 'Zhibao', ''], ['Néron', 'David', '...",2023
2,2208.11816,Bo Tang Dr.,Bo Tang and Petre Stoica,MIMO Multifunction RF Systems: Detection Perfo...,NaN,NaN,10.1109/TSP.2022.3202315,NaN,cs.IT eess.SP math.IT,http://arxiv.org/licenses/nonexclusive-distrib...,This paper studies the detection performance...,"[{'version': 'v1', 'created': 'Thu, 25 Aug 202...",2022-08-29,"[['Tang', 'Bo', ''], ['Stoica', 'Petre', '']]",2022
3,2207.03548,Hoa Nguyen Tien,"Tien Hoa Nguyen, Van Dai Do",An Analysis of Uplink Success Probability in M...,NaN,NaN,NaN,NaN,cs.IT eess.SP math.IT,http://creativecommons.org/licenses/by/4.0/,The development of the low power wide area n...,"[{'version': 'v1', 'created': 'Fri, 17 Jun 202...",2022-07-11,"[['Nguyen', 'Tien Hoa', ''], ['Do', 'Van Dai',...",2022
4,1909.10289,Jie Li,"Jie Li, David Karpuk, Camilla Hollanti",Towards Practical Private Information Retrieva...,Accepted for publication in the IEEE Transacti...,NaN,10.1109/TCOMM.2020.2980833,NaN,cs.IT math.IT,http://arxiv.org/licenses/nonexclusive-distrib...,Private information retrieval (PIR) is the p...,"[{'version': 'v1', 'created': 'Mon, 23 Sep 201...",2020-03-13,"[['Li', 'Jie', ''], ['Karpuk', 'David', ''], [...",2020


- Filter by number of tokens

In [7]:
data['num_tokens'] = data['abstract'].apply(word_tokenize).apply(len)

In [8]:
data['num_tokens'].describe()

count    8000.000000
mean      192.422875
std        61.096949
min        13.000000
25%       151.000000
50%       190.000000
75%       231.000000
max       608.000000
Name: num_tokens, dtype: float64

In [9]:
data = data[(data['num_tokens'] >= MIN_TOKENS) & (data['num_tokens'] <= MAX_TOKENS)]
data['abstract'] = data['abstract'].apply(lambda x : x.strip())
len(data)

7656

# Annotate the data

In [22]:
def annotate(
    data : pd.DataFrame, 
    th_step : float = 0.2, 
    minimum : float = 0.0,
    maximum : float = 1.0
) -> dict[int, dict]:

    documents = []
    indices = []
    l = {}
    
    for i,threshold in enumerate(np.arange(minimum, maximum, th_step)):

        for j,(idx,row) in tqdm(enumerate(data.iterrows()), total=len(data)):

            documents.append(row['abstract'])
            indices.append(idx)
                
            if (len(documents) == BATCH_SIZE) or (j == len(data) - 1):
            
                results = make_request(
                    url=BASE_URL,
                    documents=documents,
                    min_threshold=threshold,
                    k=10
                )

                try:
                    results = results.unwrap().json()

                except Exception as e:
                    print(e)
                    continue

                for idx,venues,doc in zip(indices,results,documents):
                        
                    if idx not in l:
                        
                        l[idx] = {
                            "id" : idx,
                            "abstract" : doc,
                                "venues" : {
                                    venue['name'] : { "rank" : venue['rank'], "min_th" : threshold }
                                    for venue in venues
                                }
                            }

                    else:

                        for venue in venues:
                            l[idx]['venues'][venue['name']]['min_th'] = threshold

                documents.clear()
                indices.clear()

    return l

In [23]:
annotated_data = annotate(data, th_step=0.1)

  0%|          | 0/7656 [00:00<?, ?it/s]

100%|██████████| 7656/7656 [00:32<00:00, 238.13it/s]


In [24]:
len(annotated_data)

7656

In [26]:
dic = defaultdict(int)

for record in tqdm(annotated_data.values(), total=len(annotated_data)):
    
    for keyword, k_info in record['venues'].items():
        dic[k_info['min_th']] += 1

  0%|          | 0/7656 [00:00<?, ?it/s]

100%|██████████| 7656/7656 [00:00<00:00, 246803.41it/s]


In [27]:
with open("annotated_data.json", "w") as f:
    json.dump(annotated_data, f, indent=4)

In [28]:
dic

defaultdict(int,
            {np.float64(0.9): 17534,
             np.float64(0.6000000000000001): 8035,
             np.float64(0.4): 8864,
             np.float64(0.30000000000000004): 8418,
             np.float64(0.2): 5977,
             np.float64(0.8): 9067,
             np.float64(0.5): 8557,
             np.float64(0.7000000000000001): 7788,
             np.float64(0.1): 2207,
             np.float64(0.0): 113})

In [30]:
all_keywords = set()

for record in annotated_data.values():
    all_keywords.update(record['venues'].keys())

In [31]:
print(len(all_keywords))

55
